In [10]:
# json.loads()
# The json.loads() function is a built-in Python tool used to parse a JSON-formatted string 
#and convert it into a corresponding Python native object (such as a dictionary or a list)

import json

# A valid JSON string (Note: JSON keys and strings MUST use double quotes)
json_string = '{"name": "Alice", "age": 30, "is_admin": true, "hobbies": null}'

# json_string = '{"http_method": "FETCH", "response_status_code": "200"}'
print(type(json_string))

<class 'str'>


In [11]:
data = json.loads(json_string)

print(data)
print(type(data))  
print(data["name"])  
print(data["is_admin"])  


{'name': 'Alice', 'age': 30, 'is_admin': True, 'hobbies': None}
<class 'dict'>
Alice
True


In [1]:
import json
from typing import List, Optional
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field, ValidationError

In [13]:
class APIEndpointConfig(BaseModel):
  endpoint_name: str = Field(
      description="The URI path for the endpoint (e.g. /api/v1/users)"
  )
  http_method: str = Field(
      description="HTTP method in uppercase: GET, POST, PUT, DELETE"
  )
  requires_auth: bool = Field(
      description="Whether JWT/API token authentication is required"
  )
  query_params: List[str] = Field(
      default=[], description="List of expected query parameter names"
  )
  response_status_code: int = Field(
      default=200, description="Default successful HTTP status code"
  )

Think of it like a relay race:The LLM's output (the text or JSON it generates) becomes the input to Pydantic.Pydantic's job is to inspect that raw text/JSON, check if it matches the rules, and convert it into a clean, safe Python object.So the flow looks like this:$$\text{User Prompt} \longrightarrow \text{LLM generates JSON} \longrightarrow \text{Pydantic validates that JSON}$$When we use Pydantic here, we aren't validating what the user typed; we are validating what the LLM produced to make sure it didn't make any mistakes before our backend code touches it.

#  Pillar 1, Script 2: Structured Outputs & Schemas (Pydantic & Validation Retries)


"How do you guarantee schema enforcement when requesting JSON from an LLM, and how do you handle validation failures gracefully?"



# Core Techniques in Script #2
## 1. Pydantic Schema Enforcement: Passing a Pydantic model directly into model.with_structured_output(Schema) to force the provider (OpenAI, Anthropic, etc.) to use native Function/Tool Calling under the hood for guaranteed schema formatting.

## 2.Self-Correction Retry Loop: Intercepting Pydantic ValidationError exceptions, appending the validation error message back to the LLM prompt, and re-invoking until valid JSON is returned.

In [19]:
import json
from typing import List, Optional
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field, ValidationError

In [20]:
# 1. Define Strict Pydantic Schema
class APIEndpointConfig(BaseModel):
  endpoint_name: str = Field(
      description="The URI path for the endpoint (e.g. /api/v1/users)"
  )
  http_method: str = Field(
      description="HTTP method in uppercase: GET, POST, PUT, DELETE"
  )
  requires_auth: bool = Field(
      description="Whether JWT/API token authentication is required"
  )
  query_params: List[str] = Field(
      default=[], description="List of expected query parameter names"
  )
  response_status_code: int = Field(
      default=200, description="Default successful HTTP status code"
  )

In [21]:
# 2. Simulated LLM call with Structured Output
def call_llm_with_structured_output(user_prompt: str, model=None) -> APIEndpointConfig:
  """Demonstrates how with_structured_output wraps model calls."""
  # If using real LangChain model:
  # structured_llm = model.with_structured_output(APIEndpointConfig)
  # return structured_llm.invoke(user_prompt)

  # Simulated raw output demonstration:
  raw_json_response = """
    {
        "endpoint_name": "/api/v1/analytics/query",
        "http_method": "POST",
        "requires_auth": true,
        "query_params": ["start_date", "end_date", "limit"],
        "response_status_code": 200
    }
    """
  # Parse directly into Pydantic model for verification
  validated_data = APIEndpointConfig.model_validate_json(raw_json_response)
  return validated_data

In [22]:
# 3. Robust Retry Loop for Schema Validation
def safe_extract_schema(prompt: str, max_retries: int = 3) -> APIEndpointConfig:
  """Robust wrapper that catches Pydantic validation errors and feeds them back to the LLM for correction."""
  attempt = 0
  error_history = ""

  while attempt < max_retries:
    attempt += 1
    print(f"\n--- [Attempt {attempt}/{max_retries}] Requesting Structured Output ---")

    try:
      # Inject error history if previous attempts failed
      full_prompt = prompt
      if error_history:
        full_prompt += f"\n\n[PREVIOUS VALIDATION ERROR]: {error_history}. Please fix the JSON schema."

      # Execute LLM call
      result = call_llm_with_structured_output(full_prompt)
      print("✓ Schema Validation Succeeded!")
      return result

    except ValidationError as e:
      print(f"✗ Pydantic Validation Error Caught: {e}")
      error_history = str(e)

  raise RuntimeError(
      f"Failed to produce valid structured output after {max_retries} attempts."
  )

In [23]:
# --- Execution Test ---
if __name__ == "__main__":
  user_request = "Create an API configuration for an authenticated analytics query endpoint that accepts start_date and end_date filtering."

  config_result = safe_extract_schema(user_request)

  print("\n================ VALIDATED PYDANTIC OBJECT ================")
  print(f"Endpoint Path : {config_result.endpoint_name}")
  print(f"HTTP Method   : {config_result.http_method}")
  print(f"Auth Required : {config_result.requires_auth}")
  print(f"Query Params  : {config_result.query_params}")
  print(f"Status Code   : {config_result.response_status_code}")


--- [Attempt 1/3] Requesting Structured Output ---
✓ Schema Validation Succeeded!

================ VALIDATED PYDANTIC OBJECT ================
Endpoint Path : /api/v1/analytics/query
HTTP Method   : POST
Auth Required : True
Query Params  : ['start_date', 'end_date', 'limit']
Status Code   : 200
